## Modbus IDS (unsupervised) — cascade pipeline

Logic lives in **`ids_pipeline`** at the repo root: window Isolation Forest + per-message Isolation Forest, optional **time-based train split**, and **cascade** alerts (window anomaly gates packet-level ranking or AND rule).

**Dependencies:** `pandas`, `numpy`, `scikit-learn`, `matplotlib`

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ids_pipeline import (
    CascadeConfig,
    fit_and_score,
    load_attack_intervals,
    load_clean_csv,
    packet_attack_labels,
    window_attack_overlap_labels,
)

### Paths and configuration

- **`TRAIN_FRAC`**: `None` = fit on the full capture; otherwise a value in `(0,1)` fits scalers + both forests only on rows with `ts < t_cut` and windows with `t_end <= t_cut`, then scores **all** rows. Report **test-only** metrics in a later cell when split is on.
- **`cascade_mode`**: `"top_k"` — within each window flagged by the window IF, mark the **`cascade_top_k`** lowest `pkt_if_score` messages; `"and"` — require packet IF outlier too.

In [ ]:
DATA_DIR = ROOT / "Initial Dataset 4-14"
CSV_PATH = DATA_DIR / "modbus_dataset_2026-04-14 13_17_14.116610.csv"
JSONL_PATHS = [
    DATA_DIR / "ids_events_orch1.jsonl",
    DATA_DIR / "ids_events_orch2.jsonl",
    DATA_DIR / "ids_events_orch3.jsonl",
]

EVAL_TS_OFFSET_SEC = 0.0  # add to packet times if JSONL and CSV clocks disagree

cfg = CascadeConfig(
    window_sec=10.0,
    include_dst_context=True,
    drop_src_identity=True,
    win_contamination="auto",
    pkt_contamination=0.01,
    n_estimators_win=300,
    n_estimators_pkt=150,
    packet_include_dst=False,
    train_frac=0.7,
    cascade_mode="top_k",
    cascade_top_k=25,
)

df0 = load_clean_csv(CSV_PATH)
res = fit_and_score(df0, cfg)
df, feat, pkt = res.df, res.feat, res.pkt
print("t_cut:", res.t_cut)
df.head()

### Quick counts

In [ ]:
n_win = len(feat)
n_win_anom = int((feat["if_pred"] == -1).sum())
n_pkt = len(pkt)
n_pkt_anom = int((pkt["pkt_if_pred"] == -1).sum())
n_cascade = int(pkt["cascade_alert"].sum())
print(f"Windows: {n_win} | window IF anomalies: {n_win_anom}")
print(f"Messages: {n_pkt} | packet IF anomalies: {n_pkt_anom}")
print(f"Cascade alerts (mode={cfg.cascade_mode}): {n_cascade}")

### Window IF score vs time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(feat["t_start"], feat["if_score"], lw=0.8, color="0.65", label="score (all windows)")
anomaly_mask = feat["if_pred"] == -1
if anomaly_mask.any():
    ax.scatter(
        feat.loc[anomaly_mask, "t_start"],
        feat.loc[anomaly_mask, "if_score"],
        color="crimson",
        s=22,
        zorder=5,
        label="anomaly (if_pred=-1)",
    )
if res.t_cut is not None:
    ax.axvline(res.t_cut, color="navy", ls="--", lw=1, label="train | test cut")
ax.set_title("Window Isolation Forest (higher = more normal)")
ax.set_xlabel("epoch seconds (from CSV)")
ax.set_ylabel("if_score")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

### Per-message score histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(pkt["pkt_if_score"], bins=80, color="0.4", edgecolor="white")
ax.set_xlabel("pkt_if_score (higher = more normal)")
ax.set_ylabel("count")
ax.set_title("Per-message Isolation Forest scores")
plt.tight_layout()
plt.show()

### Most anomalous messages (global)

In [ ]:
show_pkt = ["Time", "Dst IP", "Func", "Type", "Unit_ID", "Trans_ID", "iat_sec", "data_len", "pkt_if_score", "pkt_if_pred", "win_if_pred", "cascade_alert"]
show_pkt = [c for c in show_pkt if c in pkt.columns]
worst = pkt.sort_values("pkt_if_score", ascending=True).head(40)
print(worst[show_pkt].to_string())

### Evaluation vs JSONL (episode intervals)

Ground truth: packet timestamp inside any **`episode_start`–`episode_end`** interval. Compare **packet IF only** vs **cascade** flags. When `train_frac` is set, metrics are shown on **all** data and on **test** rows (`ts >= t_cut`) separately.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

attack_intervals = load_attack_intervals(JSONL_PATHS)
print(f"Loaded {len(attack_intervals)} attack intervals")


def eval_packet(name, y_true, y_pred, score_anom):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    print(f"\n=== {name} ===")
    print(f"positives (in interval): {int(y_true.sum())} / {len(y_true)} ({100 * y_true.mean():.3f}%)")
    print(f"predicted alerts:         {int(y_pred.sum())} / {len(y_pred)} ({100 * y_pred.mean():.3f}%)")
    print("confusion [true x pred]:", confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist())
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["not_in_interval", "in_attack_interval"],
            digits=3,
            zero_division=0,
        )
    )
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    print(f"Binary P/R/F1 (alert=1): P={p:.4f} R={r:.4f} F1={f1:.4f}")
    if y_true.min() < y_true.max():
        try:
            print(f"ROC-AUC: {roc_auc_score(y_true, score_anom):.4f}")
            print(f"Average precision: {average_precision_score(y_true, score_anom):.4f}")
        except ValueError as e:
            print("ROC/AP:", e)


if not attack_intervals:
    print("No intervals — check JSONL paths.")
else:
    ts_eval = pkt["ts"].values.astype(np.float64) + float(EVAL_TS_OFFSET_SEC)
    y_true = packet_attack_labels(ts_eval, attack_intervals, offset_sec=0.0)
    score_anom = -pkt["pkt_if_score"].values.astype(np.float64)
    y_pkt = (pkt["pkt_if_pred"].values == -1).astype(np.int8)
    y_cas = pkt["cascade_alert"].values.astype(np.int8)

    eval_packet("Packet IF only (all rows)", y_true, y_pkt, score_anom)
    eval_packet("Cascade alerts (all rows)", y_true, y_cas, score_anom)

    if res.t_cut is not None:
        m = pkt["ts"].values >= res.t_cut
        eval_packet("Packet IF (test rows only)", y_true[m], y_pkt[m], score_anom[m])
        eval_packet("Cascade (test rows only)", y_true[m], y_cas[m], score_anom[m])

    off = float(EVAL_TS_OFFSET_SEC)
    yw_true = window_attack_overlap_labels(feat, attack_intervals, offset_sec=off)
    yw_pred = (feat["if_pred"].values == -1).astype(np.int8)
    print("\n=== Window IF vs interval overlap ===")
    print(confusion_matrix(yw_true, yw_pred, labels=[0, 1]))
    print(
        classification_report(
            yw_true,
            yw_pred,
            labels=[0, 1],
            target_names=["win_ok", "win_attack_overlap"],
            digits=3,
            zero_division=0,
        )
    )